In [1]:
import madina as md
import madina.una.tools as una
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely import Point, MultiPoint, LineString, Polygon
import uuid
import plotly.express as px
import os 

In [2]:
os.chdir('/workspaces/exp_sidewalk_planning/')
from src.funcs import betweeness

Layer name           | Visible | projection | rows  | File path           
sidewalks            |       1 | EPSG:3857  |   357 | /workspaces/exp_sidewalk_planning/data/berlin/sidewalks_pseudo.geojson
buildings            |       1 | EPSG:3857  |  2860 | /workspaces/exp_sidewalk_planning/data/berlin/buildings_pseudo.geojson
kitas                |       1 | EPSG:3857  |    80 | /workspaces/exp_sidewalk_planning/data/berlin/kitas_pseudo.geojson
Geographic center: (13.396303813961305, 52.4916423315232)
No network graph yet. First, insert a layer that contains network segments (streets, sidewalks, ..) and call create_street_network(layer_name,  weight_attribute=None)
	Then,  insert origins and destinations using 'insert_nodes(label, layer_name, weight_attribute)'
	Finally, when done, create a network by calling 'create_street_network()'


In [3]:
gdf = betweeness(origin='kitas', destination='buildings', job_id=1, between_radius = 500,detour_ratio=1.2, beta = 0.001)

In [4]:
df = gpd.read_file('data/berlin/perceptions/berlin_perceptions_lor.geojson')
#df = df.to_crs(epsg=4326)

Skipping field time: unsupported OGR type: 10


In [18]:
df.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [5]:
def return_sentiment(gdf):
    gdf = gdf[gdf.betweenness > 0]
    gdf['geometry'] = gdf.buffer(3)
    gdf = gpd.sjoin(predicate='intersects', left_df=gdf, right_df=df, how='left')
    gdf = gdf[gdf.choice == 'left']
    sentiment = gdf[gdf['choice'] == 'left'].groupby('study_question').count()['bezirk']
    return sentiment

In [6]:
sentiment = return_sentiment(gdf)

/usr/local/lib/python3.11/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [7]:
sentiment

study_question
livelier           36
more beautiful     13
more boring        11
more depressing     5
safer              37
wealthier          24
Name: bezirk, dtype: int64